In [35]:
import numpy as np
from os.path import join, exists
import os
from o2_utils.selectors import find_files_from_pattern
from datetime import datetime
from glob import glob
import pandas as pd
from pathlib import Path

In [2]:
def days_between(s1, s2):
    date1 = datetime.strptime(s1.split('_')[0], '%Y%m%d')
    date2 = datetime.strptime(s2.split('_')[0], '%Y%m%d')
    delta = abs(date2 - date1)
    return delta.days



In [27]:
data_path = "/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ"
mice = ["Male_03"]
videos = []
for mouse in mice:
    videos += sorted(find_files_from_pattern(join(data_path, f'{mouse}/2024*/2024*camera_acquisition/2024*'), '*.mp4', exclude_patterns=["azure", "TRIM", "xxx", "COMPRESSED"], error_behav="pass"))
    videos = [v for v in videos if "xxx" not in v.lower()]
    calibration_folders = sorted(find_files_from_pattern(join(data_path, f'{mouse}/2024*/'), '2024*calibration', exclude_patterns=["azure", "TRIM", "xxx", "COMPRESSED"], error_behav="pass"))


In [41]:
recording_prefixes = {v: os.path.basename(v.split('.top')[0]) for v in videos if (v.endswith('.top.0.mp4') or v.endswith('.top.mp4'))}
calib_prefixes = {os.path.basename(c): c for c in calibration_folders}
calib_prefixes_list = list(calib_prefixes.keys())
calib_mapping = {}
for r, r_prefix in recording_prefixes.items():
    diffs = [days_between(r_prefix, c_prefix) for c_prefix in calib_prefixes.keys()]
    calib_mapping[Path(r).parent] = Path(calib_prefixes[calib_prefixes_list[np.argmin(diffs)]])

In [42]:
calib_mapping

{PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition/20240723_chronic-no-pertub_male-03_24-07-23-13-51-47-308105'): PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_calibration'),
 PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition/20240723_chronic-no-pertub_male-03_24-07-23-13-54-52-420754'): PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_calibration'),
 PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition/20240723_chronic-no-pertub_male-03_24-07-23-14-16-19-742335'): PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_calibration'),
 PosixPath('/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition/20240723_chronic-no-pertub_male-03_24-07-23-14-38-09-463745'): 

In [48]:
# airflow spreadsheet has these cols:
# we want to generate a csv we can open and then copy into the airflow spreadsheet
"""
Subject	duration_m	video_recording_id	ephys_id	calibration_id	video_location_on_o2	ephys_location_on_o2	calibration_location_on_o2	samplerate	username	n_ephys_streams	max_video_duration_m	use_local	User
J04502	30	20240426_J04502			/n/groups/datta/Jonah/20240206_methimazole_sniffing/20240206_6cam/data/J04502/			120	jop9552	0	1000	FALSE	jonah
"""
recording_dict = {}
for recording_dir in calib_mapping:
    recording_prefix = os.path.basename(recording_dir)
    calibn_id = calib_mapping[recording_dir].name
    calibn_path = calib_mapping[recording_dir].parent

    subject = recording_prefix.split('_')[2]

    # read the metadata file and compute the median fps
    try:
        metadata_file = list(recording_dir.glob("*.top*metadata.csv"))[0]
    except IndexError:
        metadata_file = glob(join(data_path, subject, recording_prefix, f"{recording_prefix}.top.0.metadata.csv"))[0]
    df = pd.read_csv(metadata_file, header=0)
    fps = np.round(1 / (np.median(np.diff(df['frame_timestamp'].values)) / 1e9))
    duration_min = len(df) / fps / 60

    recording_dict[recording_prefix] = {
        "Subject": subject,
        "duration_m": duration_min.astype(int),
        "video_recording_id": recording_prefix,
        "ephys_id": "",
        "calibration_id": calibn_id,
        "video_location_on_o2": recording_dir.parent,
        "ephys_location_on_o2": "",
        "calibration_location_on_o2": calibn_path,
        "samplerate": int(fps),
        "username": "jop9552",
        "n_ephys_streams": 0,
        "max_video_duration_m": 1000,
        "use_local": "FALSE",
        "User": "jonah"
    }


In [49]:
df = pd.DataFrame.from_dict(recording_dict, orient='index')
df.head()

,Subject,duration_m,video_recording_id,ephys_id,calibration_id,video_location_on_o2,ephys_location_on_o2,calibration_location_on_o2,samplerate,username,n_ephys_streams,max_video_duration_m,use_local,User
20240723_chronic-no-pertub_male-03_24-07-23-13-51-47-308105,male-03,1,20240723_chronic-no-pertub_male-03_24-07-23-13...,,20240723_calibration,/n/groups/datta/charlotte/DATA/Internal_state_...,,/n/groups/datta/charlotte/DATA/Internal_state_...,120,jop9552,0,1000,FALSE,jonah
20240723_chronic-no-pertub_male-03_24-07-23-13-54-52-420754,male-03,20,20240723_chronic-no-pertub_male-03_24-07-23-13...,,20240723_calibration,/n/groups/datta/charlotte/DATA/Internal_state_...,,/n/groups/datta/charlotte/DATA/Internal_state_...,120,jop9552,0,1000,FALSE,jonah
20240723_chronic-no-pertub_male-03_24-07-23-14-16-19-742335,male-03,20,20240723_chronic-no-pertub_male-03_24-07-23-14...,,20240723_calibration,/n/groups/datta/charlotte/DATA/Internal_state_...,,/n/groups/datta/charlotte/DATA/Internal_state_...,120,jop9552,0,1000,FALSE,jonah
20240723_chronic-no-pertub_male-03_24-07-23-14-38-09-463745,male-03,20,20240723_chronic-no-pertub_male-03_24-07-23-14...,,20240723_calibration,/n/groups/datta/charlotte/DATA/Internal_state_...,,/n/groups/datta/charlotte/DATA/Internal_state_...,120,jop9552,0,1000,FALSE,jonah
20240724_chronic-vehicle_male-03_24-07-24-11-10-47-598315,male-03,40,20240724_chronic-vehicle_male-03_24-07-24-11-1...,,20240724_calibration,/n/groups/datta/charlotte/DATA/Internal_state_...,,/n/groups/datta/charlotte/DATA/Internal_state_...,120,jop9552,0,1000,FALSE,jonah


In [50]:
for index, row in df.iterrows():
    print('\t'.join(map(str, row)))

male-03	1	20240723_chronic-no-pertub_male-03_24-07-23-13-51-47-308105		20240723_calibration	/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition		/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723	120	jop9552	0	1000	FALSE	jonah
male-03	20	20240723_chronic-no-pertub_male-03_24-07-23-13-54-52-420754		20240723_calibration	/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition		/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723	120	jop9552	0	1000	FALSE	jonah
male-03	20	20240723_chronic-no-pertub_male-03_24-07-23-14-16-19-742335		20240723_calibration	/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723/20240723_camera_acquisition		/n/groups/datta/charlotte/DATA/Internal_state_MOSEQ/Male_03/20240723	120	jop9552	0	1000	FALSE	jonah
male-03	20	20240723_chronic-no-pertub_male-03_24-07-23-14-38-09-463745		20240723_calibration	/n/groups/datta/charlotte/DATA/Inte